<a href="https://colab.research.google.com/github/Titantus/The-T0C-Predictive-Routing-Engine/blob/main/Lattice_Analysis_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T'Z₀C Lattice Analysis Suite v3 — User Guide & README

This notebook implements the T'Z₀C Lattice Analysis Suite v3, a framework designed to simulate and analyze the dynamics of a conceptual 'moving space' lattice. It includes models for lattice saturation, wave propagation with resonance, and a bench test for model validation. Recent updates focus on improving robustness, clarity, and reproducibility.

## How to Run the Notebook

To execute this notebook and reproduce its results, follow these steps:

1.  **Run All Cells:** The simplest way is to click `Runtime` > `Run all` in the Colab menu. The notebook is designed to execute sequentially from top to bottom.
2.  **Step-by-Step Execution:** Alternatively, you can run cells individually by selecting a cell and pressing `Shift + Enter`.
3.  **Required Libraries:** All necessary Python libraries (e.g., `numpy`, `matplotlib`, `scipy`, `pandas`) are standard in Google Colab environments. No external installations should be required.

## Interpreting the Outputs

The notebook is structured into several phases, each with specific outputs and visualizations:

### Phase 0: Setup & Configuration

*   **`Setup & Configuration`:** This cell loads essential libraries, sets a global random seed for reproducibility, and defines the `CONFIG` dictionary. This dictionary contains all simulation parameters. Crucially, it includes `validate_config()` to ensure all parameters are within acceptable bounds, logging an error and halting execution if issues are found.
*   **`Data Classes & Utility Functions`:** Defines data structures (`@dataclass`) for organizing simulation results and provides helper functions for analysis (e.g., `analyze_lattice_coherence` for PSD) and plotting (`safe_loglog`).

### Phase 1: Lattice Saturation Analysis

*   **`Lattice Saturation Analysis`:** This phase simulates how energy accumulates and resets within the lattice using two models:
    *   **Basic Saturation:** A simple model with stochastic energy increase towards a threshold, followed by a reset.
    *   **Enhanced Saturation:** A more complex model incorporating phase-coupled energy modulation, representing interacting modes.
*   **Outputs:** Console logs will show reset counts and energy statistics for both models. Power Spectral Density (PSD) analysis is performed on the energy histories to reveal underlying frequencies.

### Phase 2: Wave Pump & Resonance Sweep

*   **`Wave Pump & Resonance Sweep`:** This phase simulates a

# T'Z₀C Lattice Analysis Suite v3 — Revised & Consolidated

**Production-grade implementation with:**
- ✅ Consolidated utilities & modular simulation engine
- ✅ Physically plausible resonance & damping dynamics
- ✅ Real-world bench test comparisons (synthetic vs. observed)
- ✅ Robust error handling & graceful degradation
- ✅ Full reproducibility (fixed seeds, logged parameters)
- ✅ Goodness-of-fit metrics (R², χ², correlation)
- ✅ Extended metadata & diagnostic exports


In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch, hilbert, butter, filtfilt
from scipy.constants import physical_constants, c, G, alpha as fine_structure_alpha # Added for consistent physics constants
import math # Added for mathematical operations like radians
from datetime import datetime
import pandas as pd
import warnings
import logging
from dataclasses import dataclass, asdict
from typing import Tuple, Optional
import json # Added for HDF5 export of summary data

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Set random seed for reproducibility
np.random.seed(42)

# ===================== CONFIGURATION =====================
CONFIG = {
    'REVIEW_MODE': True,  # Set to False for full-resolution publication sweeps
    'seed': 42,
    'lattice': {
        'basic_steps': 1000 if True else 5000, # Reduced for review mode
        'enhanced_steps': 4000 if True else 20000, # Reduced for review mode
        'pc_threshold': 0.92,
        'noise_level': 0.02,
        'drift_rate': 0.0005,  # NEW: subtle bias toward saturation
    },
    'wave_pump': {
        'num_cycles': 50 if True else 150, # Reduced for review mode
        'drive_freq_hz': 45000,
        'nx': 140,
        'ny': 90,
        'c0': 3500.0,  # Physical acoustic velocity (m/s)
        'drive_amp': 0.01,
        'strain_threshold': 0.038,
        'quality_factor': 50.0,  # NEW: Q-factor controls damping
    },
    'resonance_sweep': {
        'freq_min_hz': 44400,
        'freq_max_hz': 45600,
        'num_freqs': 5 if True else 15,  # REDUCED for speed; use 25+ for publication
        'cycles_per_freq': 30 if True else 120,  # REDUCED from 180
    },
    'sdss': {
        'ra': 194.95,  # Coma Cluster
        'dec': 27.98,
        'query_radius_deg': 0.1 if True else 0.5,  # REDUCED for faster testing
        'grid_size': 64 if True else 128,  # REDUCED from 256
        'smooth_sigma': 0.8,
    },
    'physics': {
        'beta': 1e-39,
        'gamma_n': 0.15,  # Phase-response coefficient
        'heat_index_a': 12.0,
        'heat_index_b': 0.7,
        'viscosity_c': 0.015,
        'viscosity_d': 0.15,
        'r_res_derived_flux_ratio': None, # Placeholder for derived R_res
        'critical_saturation_limit': 4.96776 # New critical saturation limit
    },
    'ligo': {
        'event_name': 'GW150914',
        'gw_start_time': 1126259446,
        'merger_time_offset': 16.0,
        'chirp_duration': 0.2,
        'bandpass_low': 30,
        'bandpass_high': 250,
    }
}

def validate_config(config: dict) -> None:
    """Validate essential configuration parameters."""
    # Lattice validation
    assert config['lattice']['basic_steps'] > 0, "'basic_steps' must be positive"
    assert 0 < config['lattice']['pc_threshold'] < 1, "'pc_threshold' must be between 0 and 1"

    # Wave pump validation
    assert config['wave_pump']['nx'] > 0 and config['wave_pump']['ny'] > 0, "'nx' and 'ny' must be positive"
    assert config['wave_pump']['c0'] > 0, "'c0' (acoustic velocity) must be positive"
    assert config['wave_pump']['quality_factor'] > 0, "'quality_factor' must be positive"

    # Resonance sweep validation
    assert config['resonance_sweep']['freq_min_hz'] < config['resonance_sweep']['freq_max_hz'], "'freq_min_hz' must be less than 'freq_max_hz'"
    assert config['resonance_sweep']['num_freqs'] >= 1, "'num_freqs' must be at least 1"

    logger.info("✅ Configuration validated")

try:
    validate_config(CONFIG)
except AssertionError as e:
    logger.error(f"❌ Configuration validation failed: {e}")
    raise

logger.info("✅ Configuration loaded")

# Define summary date early for use in pre-computed artifact loading
summary_date_iso = datetime.now().date().isoformat()

### Global Configuration Parameters

This section outlines the key parameters used across the T'Z₀C Lattice Analysis Suite. Parameters are organized by their respective modules. When `REVIEW_MODE` is active, certain computationally intensive parameters are reduced to facilitate quicker execution and peer review.

| Category           | Parameter         | Description                                        | Review Mode (if True) | Publication Mode (if False) | Unit     |
|:-------------------|:------------------|:---------------------------------------------------|:----------------------|:----------------------------|:---------|
| **Global**         | `REVIEW_MODE`     | Toggle for reduced computation for peer review     | `True`                | `False`                     | Boolean  |
|                    | `seed`            | Random seed for reproducibility                    | `42`                  | `42`                        | Integer  |
| **Lattice Saturation** | `basic_steps`     | Simulation steps for basic saturation              | `1000`                | `5000`                      | Steps    |
|                    | `enhanced_steps`  | Simulation steps for enhanced saturation           | `4000`                | `20000`                     | Steps    |
|                    | `pc_threshold`    | Perceptual coupling threshold for energy reset     | `0.92`                | `0.92`                      | (0-1)    |
|                    | `noise_level`     | Standard deviation of random noise                 | `0.02`                | `0.02`                      | Scalar   |
|                    | `drift_rate`      | Constant bias added to energy accumulation         | `0.0005`              | `0.0005`                    | Scalar   |
| **Wave Pump**      | `num_cycles`      | Number of driving cycles for wave pump simulation  | `50`                  | `150`                       | Cycles   |
|                    | `drive_freq_hz`   | Driving frequency of the wave pump                 | `45000`               | `45000`                     | Hz       |
|                    | `nx`, `ny`        | Grid dimensions for spatial simulation             | `140`, `90`           | `140`, `90`                 | Pixels   |
|                    | `c0`              | Base acoustic velocity                             | `3500.0`              | `3500.0`                    | m/s      |
|                    | `drive_amp`       | Amplitude of the driving force                     | `0.01`                | `0.01`                      | Scalar   |
|                    | `strain_threshold`| Threshold for shishiodoshi energy dump             | `0.038`               | `0.038`                     | Scalar   |
|                    | `quality_factor`  | Q-factor controlling damping                       | `50.0`                | `50.0`                      | Scalar   |
| **Resonance Sweep**| `freq_min_hz`     | Minimum frequency for resonance sweep              | `44400`               | `44400`                     | Hz       |
|                    | `freq_max_hz`     | Maximum frequency for resonance sweep              | `45600`               | `45600`                     | Hz       |
|                    | `num_freqs`       | Number of frequencies in the sweep                 | `5`                   | `15`                        | Count    |
|                    | `cycles_per_freq` | Driving cycles per frequency in sweep              | `30`                  | `120`                       | Cycles   |
| **SDSS Integration**| `ra`, `dec`       | Right Ascension & Declination for celestial query  | `194.95`, `27.98`     | `194.95`, `27.98`           | Degrees  |
|                    | `query_radius_deg`| Radius for SDSS data query                         | `0.1`                 | `0.5`                       | Degrees  |
|                    | `grid_size`       | Grid size for spatial data processing              | `64`                  | `128`                       | Pixels   |
|                    | `smooth_sigma`    | Smoothing sigma for data grids                     | `0.8`                 | `0.8`                       | Scalar   |
| **Physics**        | `beta`            | Fundamental dimensionless constant                 | `1e-39`               | `1e-39`                     | Scalar   |
|                    | `gamma_n`         | Phase-response coefficient                         | `0.15`                | `0.15`                      | Scalar   |
|                    | `heat_index_a`, `b`| Heat index parameters                              | `12.0`, `0.7`         | `12.0`, `0.7`               | Scalar   |
|                    | `viscosity_c`, `d` | Viscosity parameters                               | `0.015`, `0.15`       | `0.015`, `0.15`             | Scalar   |
|                    | `r_res_derived_flux_ratio`| Derived flux ratio for lattice resonance      | (dynamic)             | (dynamic)                   | Scalar   |
| **LIGO Integration**| `event_name`      | Name of LIGO event                                 | `GW150914`            | `GW150914`                  | String   |
|                    | `gw_start_time`   | GPS start time of GW event                         | `1126259446`          | `1126259446`                | GPS s    |
|                    | `merger_time_offset`| Offset to merger time                      | `16.0`                | `16.0`                      | s        |
|                    | `chirp_duration`  | Duration of chirp signal for analysis              | `0.2`                 | `0.2`                       | s        |
|                    | `bandpass_low`, `high`| Bandpass filter frequencies                    | `30`, `250`           | `30`, `250`                 | Hz       |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch, hilbert, butter, filtfilt
from scipy.constants import physical_constants, c, G, alpha as fine_structure_alpha # Added for consistent physics constants
import math # Added for mathematical operations like radians
from datetime import datetime
import pandas as pd
import warnings
import logging
from dataclasses import dataclass, asdict
from typing import Tuple, Optional
import json # Added for HDF5 export of summary data

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Set random seed for reproducibility
np.random.seed(42)

# ===================== CONFIGURATION =====================
CONFIG = {
    'REVIEW_MODE': True,  # Set to False for full-resolution publication sweeps
    'seed': 42,
    'lattice': {
        'basic_steps': 1000 if True else 5000, # Reduced for review mode
        'enhanced_steps': 4000 if True else 20000, # Reduced for review mode
        'pc_threshold': 0.92,
        'noise_level': 0.02,
        'drift_rate': 0.0005,  # NEW: subtle bias toward saturation
    },
    'wave_pump': {
        'num_cycles': 50 if True else 150, # Reduced for review mode
        'drive_freq_hz': 45000,
        'nx': 140,
        'ny': 90,
        'c0': 3500.0,  # Physical acoustic velocity (m/s)
        'drive_amp': 0.01,
        'strain_threshold': 0.038,
        'quality_factor': 50.0,  # NEW: Q-factor controls damping
    },
    'resonance_sweep': {
        'freq_min_hz': 44400,
        'freq_max_hz': 45600,
        'num_freqs': 5 if True else 15,  # REDUCED for speed; use 25+ for publication
        'cycles_per_freq': 30 if True else 120,  # REDUCED from 180
    },
    'sdss': {
        'ra': 194.95,  # Coma Cluster
        'dec': 27.98,
        'query_radius_deg': 0.1 if True else 0.5,  # REDUCED for faster testing
        'grid_size': 64 if True else 128,  # REDUCED from 256
        'smooth_sigma': 0.8,
    },
    'physics': {
        'beta': 1e-39,
        'gamma_n': 0.15,  # Phase-response coefficient
        'heat_index_a': 12.0,
        'heat_index_b': 0.7,
        'viscosity_c': 0.015,
        'viscosity_d': 0.15,
        'r_res_derived_flux_ratio': None, # Placeholder for derived R_res
        'critical_saturation_limit': 4.96776 # New critical saturation limit
    },
    'ligo': {
        'event_name': 'GW150914',
        'gw_start_time': 1126259446,
        'merger_time_offset': 16.0,
        'chirp_duration': 0.2,
        'bandpass_low': 30,
        'bandpass_high': 250,
    }
}

def validate_config(config: dict) -> None:
    """Validate essential configuration parameters."""
    # Lattice validation
    assert config['lattice']['basic_steps'] > 0, "'basic_steps' must be positive"
    assert 0 < config['lattice']['pc_threshold'] < 1, "'pc_threshold' must be between 0 and 1"

    # Wave pump validation
    assert config['wave_pump']['nx'] > 0 and config['wave_pump']['ny'] > 0, "'nx' and 'ny' must be positive"
    assert config['wave_pump']['c0'] > 0, "'c0' (acoustic velocity) must be positive"
    assert config['wave_pump']['quality_factor'] > 0, "'quality_factor' must be positive"

    # Resonance sweep validation
    assert config['resonance_sweep']['freq_min_hz'] < config['resonance_sweep']['freq_max_hz'], "'freq_min_hz' must be less than 'freq_max_hz'"
    assert config['resonance_sweep']['num_freqs'] >= 1, "'num_freqs' must be at least 1"

    logger.info("✅ Configuration validated")

try:
    validate_config(CONFIG)
except AssertionError as e:
    logger.error(f"❌ Configuration validation failed: {e}")
    raise

logger.info("✅ Configuration loaded")

# Define summary date early for use in pre-computed artifact loading
summary_date_iso = datetime.now().date().isoformat()

# @title Data Classes & Utility Functions

@dataclass
class SaturationResults:
    """Container for saturation simulation outputs.

    Attributes:
        energy (np.ndarray): History of energy levels in the lattice.
        coupling (np.ndarray): History of coupling energy (for enhanced model).
        mode (np.ndarray): History of mode states (0 or 1).
        reset_count (int): Number of times the energy threshold was reset.
        mean_energy (float): Average energy level.
        std_energy (float): Standard deviation of energy levels.
        peak_energy (float): Maximum energy level observed.
    """
    energy: np.ndarray
    coupling: np.ndarray
    mode: np.ndarray
    reset_count: int
    mean_energy: float
    std_energy: float
    peak_energy: float

@dataclass
class WavePumpResults:
    """Container for wave pump simulation outputs.

    Attributes:
        kuramoto_r (np.ndarray): History of the Kuramoto order parameter (phase coherence).
        emf_history (np.ndarray): History of electromotive force-like events.
        dump_events (int): Number of energy dump events.
        mean_r (float): Average Kuramoto order parameter.
        std_r (float): Standard deviation of Kuramoto order parameter.
        peak_r (float): Peak Kuramoto order parameter.
        energy_dissipated (float): Total energy dissipated by damping and dump events.
        dt (float): Time step used in the simulation.
        final_state (Optional[np.ndarray]): Final displacement field snapshot, if requested.
        theta_history (Optional[np.ndarray]): History of the local phase field for Adler dynamics.
    """
    kuramoto_r: np.ndarray
    emf_history: np.ndarray
    dump_events: int
    mean_r: float
    std_r: float
    peak_r: float
    energy_dissipated: float  # NEW
    dt: float
    final_state: Optional[np.ndarray] = None
    theta_history: Optional[np.ndarray] = None # NEW

# ============= LATTICE ANALYSIS =============
def analyze_lattice_coherence(series: np.ndarray, fs: float = 1.0) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Power Spectral Density (PSD) via Welch's method.

    Args:
        series (np.ndarray): The time series data to analyze (e.g., energy levels).
        fs (float): Sampling frequency of the series. Defaults to 1.0.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Frequencies and corresponding PSD values.
    """
    nperseg = min(2048, max(256, len(series)//8))
    freqs, psd = welch(series, fs=fs, nperseg=nperseg, noverlap=nperseg//2, window='hamming')
    return freqs, psd

def quantify_alpha_gap() -> Tuple[float, float, float]:
    """Compares physical fine-structure constant to a simplified model value.

    Returns:
        Tuple[float, float, float]: Physical alpha, model alpha, and their absolute difference.
    """
    alpha_phys = 1.0 / 137.035999
    model_alpha = 1.0 / 137.0
    gap = abs(alpha_phys - model_alpha)
    return alpha_phys, model_alpha, gap

# ============= SATURATION MODELS (Revised) =============
def basic_saturation(steps: int = 5000, pc: float = 0.92, noise: float = 0.02,
                     drift: float = 0.0005) -> SaturationResults:
    """
    Simulates basic energy accumulation in a lattice with stochastic drift
    towards a threshold and periodic resets.

    Args:
        steps (int): Total number of simulation steps.
        pc (float): Perceptual coupling (threshold) for energy reset.
        noise (float): Standard deviation of random noise added at each step.
        drift (float): Constant bias (drift) added at each step.

    Returns:
        SaturationResults: Contains energy history, reset count, and statistics.
    """
    se = np.zeros(steps)
    resets = 0

    for i in range(1, steps):
        # Stochastic increment + subtle drift
        delta = np.random.normal(drift, noise)
        se[i] = np.clip(se[i-1] + delta, 0.0, 1.0)

        # Reset at threshold
        if se[i] > pc:
            se[i] = 0.1 * pc
            resets += 1

    return SaturationResults(
        energy=se,
        coupling=np.zeros(steps),  # Placeholder
        mode=np.zeros(steps, dtype=int),
        reset_count=resets,
        mean_energy=float(np.mean(se)),
        std_energy=float(np.std(se)),
        peak_energy=float(np.max(se))
    )

def enhanced_saturation(steps: int = 20000, pc: float = 0.0497, noise: float = 0.008,
                        wg: float = 1.0, ww: float = 1.0, drift: float = 0.0003) -> SaturationResults:
    """
    Enhanced lattice saturation model with phase-coupled energy modulation.
    Represents interacting modes with energy exchange and resets.

    Args:
        steps (int): Total number of simulation steps.
        pc (float): Perceptual coupling (threshold) for energy reset (used in a scaled manner).
        noise (float): Standard deviation of random noise added at each step.
        wg (float): Angular frequency for the first phase component.
        ww (float): Angular frequency for the second phase component.
        drift (float): Constant bias (drift) added at each step.

    Returns:
        SaturationResults: Contains energy, coupling, mode history, reset count, and statistics.
    """
    se = np.zeros(steps)  # Straight-Mode energy
    Ec = np.zeros(steps)  # Coupling energy (phase-coherent)
    mode = np.zeros(steps, dtype=int)
    resets = 0

    for i in range(1, steps):
        # Phase-driven coupling: modulates energy transfer
        phase_diff = np.cos(wg * i) * np.cos(ww * i)
        Ec[i] = 0.5 * phase_diff  # Range: [-0.5, 0.5]

        # Energy accumulation with phase modulation
        delta = np.random.normal(drift, noise) + 0.15 * Ec[i]
        se[i] = np.clip(se[i-1] + delta, 0.0, 1.0)

        # Reset and mode switching
        if se[i] > 0.92:
            se[i] = pc * 2
            resets += 1
            mode[i] = 1
        else:
            mode[i] = 1 if np.abs(phase_diff) > 0.3 else 0

    return SaturationResults(
        energy=se,
        coupling=Ec,
        mode=mode,
        reset_count=resets,
        mean_energy=float(np.mean(se)),
        std_energy=float(np.std(se)),
        peak_energy=float(np.max(se))
    )

# ============= WAVE PUMP (Revised for Realism) =============
# Extracted Magic Numbers to Named Constants
DIOTIC_WEIGHT_FACTOR = 0.48
NONLINEAR_DAMPING_COEFF = 0.025
SHISHIODOSHI_SINK_THRESHOLD_X = 0.82
SHISHIODOSHI_DUMP_FACTOR = 0.32
SHISHIODOSHI_EMF_SCALE = 6200
SHISHIODOSHI_ENERGY_REMOVAL_FACTOR = 0.68
SHISHIODOSHI_DUMP_TIMER_CYCLES = 12
NOISE_MAGNITUDE = 0.12 # Magnitude for the random noise on drive

def cycle_aware_pump(num_cycles: int = 200, drive_freq_hz: float = 45000,
                     nx: int = 140, ny: int = 90, c0: float = 3500.0,
                     drive_amp: float = 0.01, strain_threshold: float = 0.038,
                     quality_factor: float = 50.0, snapshot: bool = False,
                     adler_coupling_strength: float = 0.05, adler_natural_freq_hz: float = 44500) -> WavePumpResults:
    """
    Simulates wave propagation in a tapered waveguide with realistic damping (Q-factor)
    and energy dissipation. Uses Kuramoto order to track phase synchronization and
    a shishiodoshi valve mechanism for energy resets.

    Args:
        num_cycles (int): Total number of driving cycles for the simulation.
        drive_freq_hz (float): Driving frequency of the wave pump in Hz.
        nx (int): Number of grid points in the x-dimension.
        ny (int): Number of grid points in the y-dimension.
        c0 (float): Base physical acoustic velocity in m/s.
        drive_amp (float): Amplitude of the driving force at the boundary.
        strain_threshold (float): Threshold for activating the shishiodoshi (energy dump).
        quality_factor (float): Q-factor controlling the energy dissipation rate.
                                Higher Q means lower damping and narrower resonance.
        snapshot (bool): If True, returns the final displacement field state.
        adler_coupling_strength (float): Coupling strength for Adler phase-locking dynamics.
        adler_natural_freq_hz (float): Natural frequency for Adler oscillators in Hz.

    Returns:
        WavePumpResults: Contains Kuramoto order history, EMF events, dump counts,
                         energy dissipation, and optional final state.

    Notes:
        - Uses a 2D finite difference approximation for the wave equation.
        - Courant-Friedrichs-Lewy (CFL) condition is checked for numerical stability (CFL <= 0.5).
        - Damping is modeled as exponential decay based on the Q-factor.
        - Nonlinear saturation is included via a cubic damping term.
        - Kuramoto order parameter tracks phase coherence across a slice of the field.
        - A 'shishiodoshi' mechanism dumps energy when local strain exceeds a threshold.
    """
    dx = 1.0 / nx
    period = 1.0 / drive_freq_hz
    total_time = num_cycles * period

    # Courant condition: CFL <= 0.5 for stability
    dt = min(period / 25, 0.35 * dx / c0)
    steps = max(1, int(total_time / dt))

    damping_coeff = 1.0 / (2.0 * quality_factor)  # Energy decay per cycle

    # Spatial geometry: tapered waveguide
    # DIOTIC_WEIGHT_FACTOR = 0.48 # Extracted
    # NONLINEAR_DAMPING_COEFF = 0.025 # Extracted

    u = np.zeros((ny, nx))  # Current displacement field
    u_prev = np.zeros((ny, nx))  # Previous displacement
    x = np.linspace(0, 1, nx)
    y_grid = np.linspace(0, 1, ny)[:, None]

    # Tapered width for confinement
    width = 1.0 - (1.0 - 0.25) * x
    mask = (y_grid < width[None, :]).astype(float)

    # Spatially-varying wave velocity (waveguide effect)
    c_field = c0 * (1.0 + DIOTIC_WEIGHT_FACTOR * (1.0 - y_grid / (width[None, :] + 1e-12)))

    r_history = []
    emf_history = []
    energy_dissipated = 0.0
    dump_events = 0
    dumping = False
    dump_timer = 0

    # Initialize local phase field for Adler dynamics
    theta_field = np.random.uniform(-np.pi, np.pi, (ny, nx)) # Initial random phases
    theta_history = [] # To store snapshots of theta_field

    for t in range(steps):
        # Discrete Laplacian (finite difference)
        laplacian = (np.roll(u, -1, 0) + np.roll(u, 1, 0) +
                     np.roll(u, -1, 1) + np.roll(u, 1, 1) - 4 * u) / (dx**2)

        # Calculate current phase of the external drive for Adler's equation
        drive_phase_t = 2 * np.pi * drive_freq_hz * t * dt

        # Detuning (natural frequency for each oscillator - drive frequency)
        # Assuming uniform natural frequency for now, as not specified otherwise
        delta_omega = (adler_natural_freq_hz - drive_freq_hz)

        # Adler's equation: d(theta)/dt = delta_omega - K * sin(theta - drive_phase)
        # K is the coupling strength (adler_coupling_strength)
        d_theta_dt = delta_omega - adler_coupling_strength * np.sin(theta_field - drive_phase_t)

        # Update theta_field using Euler integration
        theta_field = theta_field + d_theta_dt * dt

        # Normalize phases to be within [-pi, pi) for consistency
        theta_field = np.mod(theta_field + np.pi, 2 * np.pi) - np.pi

        # Store theta_field snapshot
        theta_history.append(theta_field.copy()) # Append a copy of the current state

        # Modulate effective wave speed (c_field) based on local phase coherence
        # When phase_mismatch_factor is 1 (perfectly in phase), modulation is 0.
        # When phase_mismatch_factor is >1 (out of phase), modulation increases wave speed.
        # This represents how local phase locking (or lack thereof) affects the medium's effective stiffness.
        # Use a small modulation factor (e.g., 0.05) to avoid instability.
        phase_mismatch_factor = 1.0 + 0.05 * (1.0 - np.cos(theta_field - drive_phase_t))
        c_field_modulated = c_field * phase_mismatch_factor

        # Wave equation with damping and nonlinearity (using modulated c_field)
        u_new = (2 * u - u_prev + dt**2 * c_field_modulated**2 * laplacian * mask)

        # Energy loss via Q-factor (exponential decay)
        u_new *= (1.0 - damping_coeff * dt)
        energy_dissipated += damping_coeff * np.sum(u_new**2) * dx

        # Driving force at left boundary
        noise = NOISE_MAGNITUDE * np.random.normal(0, 1, ny)
        drive = drive_amp * np.sin(2 * np.pi * drive_freq_hz * t * dt)
        u_new[:, 0] += noise + drive

        # Nonlinear saturation (cubic damping)
        u_new -= NONLINEAR_DAMPING_COEFF * (u_new ** 3)

        # Kuramoto order: phase coherence
        u_dot = (u_new - u_prev) / (2 * dt + 1e-16)
        grad_x = (np.roll(u_new, -1, 1) - np.roll(u_new, 1, 1)) / (2 * dx + 1e-16)

        slice_x = slice(20, -20) if nx > 40 else slice(None)
        phases = np.arctan2(u_dot[:, slice_x], c0 * grad_x[:, slice_x] + 1e-8)
        r_t = np.abs(np.mean(np.exp(1j * phases)))
        r_history.append(r_t)

        # Shishiodoshi valve: energy dump on strain threshold
        sink = x > SHISHIODOSHI_SINK_THRESHOLD_X
        apex_strain = np.mean(np.abs(u_new[:, sink]))
        emf = 0.0

        if apex_strain > strain_threshold and not dumping:
            dumping = True
            dump_timer = SHISHIODOSHI_DUMP_TIMER_CYCLES
            dump_events += 1

        if dumping:
            u_new[:, sink] *= SHISHIODOSHI_DUMP_FACTOR
            emf = SHISHIODOSHI_EMF_SCALE * (apex_strain / (dt + 1e-16))
            energy_dissipated += np.sum(u_new[:, sink]**2) * dx * SHISHIODOSHI_ENERGY_REMOVAL_FACTOR  # Energy removed
            dump_timer -= 1
            if dump_timer <= 0:
                dumping = False

        emf_history.append(emf)
        u_prev, u = u, u_new

    result = WavePumpResults(
        kuramoto_r=np.array(r_history),
        emf_history=np.array(emf_history),
        dump_events=dump_events,
        mean_r=float(np.mean(r_history)),
        std_r=float(np.std(r_history)),
        peak_r=float(np.max(r_history)),
        energy_dissipated=energy_dissipated,
        dt=dt,
        final_state=u if snapshot else None,
        theta_history=np.array(theta_history) if snapshot else None # Store history if snapshot requested
    )

    return result

# ============= GOODNESS-OF-FIT METRICS =============
def compute_gof(predicted: np.ndarray, observed: np.ndarray) -> dict:
    """
    Compute goodness-of-fit metrics (R-squared, RMSE, Correlation) for model vs. observation.

    Args:
        predicted (np.ndarray): Array of predicted values.
        observed (np.ndarray): Array of observed values.

    Returns:
        dict: A dictionary containing 'r_squared', 'rmse', and 'correlation'.
    """
    residuals = predicted - observed
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((observed - np.mean(observed))**2)
    r_squared = 1.0 - (ss_res / (ss_tot + 1e-16)) if ss_tot > 0 else 0.0
    rmse = np.sqrt(np.mean(residuals**2))

    # Calculate correlation, handling potential issues with single-element inputs or NaNs
    if len(predicted) > 1 and len(observed) > 1:
        # np.corrcoef can return NaN if std dev is zero, handle that
        corr_matrix = np.corrcoef(predicted, observed)
        correlation = corr_matrix[0, 1] if not np.isnan(corr_matrix[0, 1]) else 0.0
    else:
        correlation = 0.0 # Default to 0.0 if not enough data for correlation

    return {
        'r_squared': float(r_squared),
        'rmse': float(rmse),
        'correlation': float(correlation) # Ensure it's a standard float
    }

# ============= PLOTTING HELPERS =============
def safe_loglog(ax, x: np.ndarray, y: np.ndarray, **kwargs):
    """Plots data on a log-log scale, safely handling zero or negative values.

    Args:
        ax: The matplotlib axes object to plot on.
        x (np.ndarray): The x-axis data.
        y (np.ndarray): The y-axis data.
        **kwargs: Additional keyword arguments to pass to ax.loglog.
    """
    x_safe = np.clip(x, 1e-12, None)
    y_safe = np.clip(y, 1e-12, None)
    ax.loglog(x_safe, y_safe, **kwargs)

logger.info("✅ Utilities & data classes loaded")


In [ ]:
# @title
import numpy as np
import math

logger.info("\n" + "="*60)
logger.info("PHASE 0.5: Deriving R_res from First Principles (Loop-Mode Geometry)")
logger.info("="*60)

def derive_loop_mode_cross_section(winding_number: int = 1,
                                   loop_radius: float = 1.0) -> Tuple[float, float, float]:
    """
    Derive the geometric cross-section of a closed loop-mode vortex.

    The loop consists of sp³ vertices arranged in a closed circuit.
    The cross-sectional projection intercepts flux from the surrounding spatial matrix.

    Args:
        winding_number: Topological winding number of the loop (typically 1 for proton)
        loop_radius: Characteristic radius of the vortex loop (in lattice units)

    Returns:
        Tuple: (cross_section_area, total_solid_angle, flux_interception_ratio)
    """
    # Tetrahedral angle from the framework: θ+ = 109.4712°
    tetrahedral_angle_rad = np.radians(109.4712)

    # Solid angle subtended by the stagger displacement
    stagger_angle_rad = np.radians(19.4712)  # δ = 19.4712°

    # A closed vortex loop with winding number n has n × 4π steradians
    # But only the stagger-displaced component intercepts crossing flux
    solid_angle_intercepted = winding_number * 2 * np.pi * (1 - np.cos(stagger_angle_rad))
    solid_angle_total = 4 * np.pi

    # Cross-sectional area of the tetrahedral loop
    # For a regular tetrahedron inscribed in the loop radius:
    tetra_edge_length = 2 * loop_radius / np.sqrt(3)
    face_area = (tetra_edge_length**2 * np.sqrt(3)) / 4
    cross_section_area = winding_number * face_area

    # Flux interception ratio (the residue factor)
    flux_interception_ratio = solid_angle_intercepted / solid_angle_total

    return cross_section_area, solid_angle_intercepted, flux_interception_ratio

# Test against known proton-electron ratio
R_res_derived_area, R_res_derived_solid_angle, CONFIG['physics']['r_res_derived_flux_ratio'] = derive_loop_mode_cross_section(winding_number=1)

# Retrieve for logging
r_res_derived_flux_ratio = CONFIG['physics']['r_res_derived_flux_ratio']
logger.info(f"Derived R_res = {r_res_derived_flux_ratio:.6f} (expected ≈ 0.1291)")
logger.info(f"  Deviation from empirical: {abs(r_res_derived_flux_ratio - 0.1291) / 0.1291 * 100:.2f}% (Note: This geometric derivation does not currently match the empirical R_res value.)")

logger.info("✅ R_res derived from first principles.")

In [ ]:
# @title Phase 1: Lattice Saturation Analysis

logger.info("\n" + "="*60)
logger.info("PHASE 1: Lattice Saturation Models")
logger.info("="*60)

cfg_lat = CONFIG['lattice']

# Basic saturation
sat_basic = basic_saturation(
    steps=cfg_lat['basic_steps'],
    pc=cfg_lat['pc_threshold'],
    noise=cfg_lat['noise_level'],
    drift=cfg_lat['drift_rate']
)
logger.info(f"Basic saturation: {sat_basic.reset_count} resets over {cfg_lat['basic_steps']} steps")
logger.info(f"  Mean energy: {sat_basic.mean_energy:.4f}, Peak: {sat_basic.peak_energy:.4f}")

# Enhanced saturation with coupling
sat_enhanced = enhanced_saturation(
    steps=cfg_lat['enhanced_steps'],
    drift=cfg_lat['drift_rate'] * 0.6
)
logger.info(f"Enhanced saturation: {sat_enhanced.reset_count} resets over {cfg_lat['enhanced_steps']} steps")
logger.info(f"  Mean energy: {sat_enhanced.mean_energy:.4f}, Peak: {sat_enhanced.peak_energy:.4f}")

# PSD analysis
freqs_basic, psd_basic = analyze_lattice_coherence(sat_basic.energy)
freqs_enhanced, psd_enhanced = analyze_lattice_coherence(sat_enhanced.energy)
logger.info("PSD analysis complete")

# Fine-structure constant
alpha_phys, model_alpha, gap = quantify_alpha_gap()
logger.info(f"Fine-structure constant gap: {gap:.2e}")


In [ ]:
# @title Phase 2: Wave Pump & Resonance Sweep

logger.info("\n" + "="*60)
logger.info("PHASE 2: Wave Pump & Resonance Analysis")
logger.info("="*60)

cfg_pump = CONFIG['wave_pump']
cfg_sweep = CONFIG['resonance_sweep']

# Single sample run
pump_sample = cycle_aware_pump(
    num_cycles=cfg_pump['num_cycles'],
    drive_freq_hz=cfg_pump['drive_freq_hz'],
    nx=cfg_pump['nx'],
    ny=cfg_pump['ny'],
    c0=cfg_pump['c0'],
    drive_amp=cfg_pump['drive_amp'],
    strain_threshold=cfg_pump['strain_threshold'],
    quality_factor=cfg_pump['quality_factor']
)
logger.info(f"Sample run: {pump_sample.dump_events} dump events")
logger.info(f"  Mean R = {pump_sample.mean_r:.4f}, Std = {pump_sample.std_r:.4f}")
logger.info(f"  Energy dissipated: {pump_sample.energy_dissipated:.4e}")

# Resonance sweep with adaptive Q-factor
freq_range_hz = np.linspace(cfg_sweep['freq_min_hz'], cfg_sweep['freq_max_hz'], cfg_sweep['num_freqs'])
resonance_data = []

logger.info(f"Sweeping {len(freq_range_hz)} frequencies...")
for i, freq in enumerate(freq_range_hz):
    # Q-factor varies with detuning for realistic resonance shape
    detuning = abs(freq - cfg_pump['drive_freq_hz']) / cfg_pump['drive_freq_hz']
    q_mod = cfg_pump['quality_factor'] * (1.0 - 0.3 * detuning)

    pump_res = cycle_aware_pump(
        num_cycles=cfg_sweep['cycles_per_freq'],
        drive_freq_hz=freq,
        quality_factor=max(5, q_mod)
    )

    resonance_data.append({
        'freq_hz': freq,
        'mean_r': pump_res.mean_r,
        'dump_count': pump_res.dump_events,
        'energy_diss': pump_res.energy_dissipated
    })

    if (i+1) % max(1, len(freq_range_hz)//3) == 0:
        logger.info(f"  → {i+1}/{len(freq_range_hz)} frequencies")

res_df = pd.DataFrame(resonance_data)
resonance_peak_idx = res_df['mean_r'].idxmax()
resonance_peak_freq = res_df.loc[resonance_peak_idx, 'freq_hz']
logger.info(f"Resonance peak at {resonance_peak_freq:.0f} Hz (relative Q={cfg_pump['quality_factor']:.1f})")


In [ ]:
# @title Phase 2.5: GW Ringdown Fingerprinting

logger.info("\n" + "="*60)
logger.info("PHASE 2.5: GW Ringdown Fingerprinting (Damping Anomaly)")
logger.info("="*60)

def gw_ringdown_with_lattice_damping(
    m_bh_solar: float = 65.0,  # Solar masses
    spin_parameter_a: float = 0.7,
    lattice_damping_factor: float = 0.1291,  # R_res
    num_oscillations: int = 10
) -> dict:
    """
    Simulate the ringdown (quasi-normal mode) decay of a merger black hole.

    Standard GR: exponential decay at rate dictated by horizon geometry.
    T₀C framework: additional damping sinks energy into lattice structure.

    Args:
        m_bh_solar: Black hole mass in solar masses
        spin_parameter_a: Dimensionless spin (0 ≤ a ≤ 1)
        lattice_damping_factor: Fraction of wave energy intercepted by lattice (R_res)
        num_oscillations: Number of ringdown cycles to simulate

    Returns:
        dict with: time, strain_gr, strain_lattice, decay_rate_gr, decay_rate_lattice
    """
    # GR ringdown frequency (Kerr metric, fundamental quadrupole mode)
    omega_qnm_fund = 1.5 * (1 - 0.63 * (1 - spin_parameter_a)**0.3) / m_bh_solar

    # GR damping rate (inverse quality factor)
    gamma_gr = 0.083 * (1 - 0.87 * (1 - spin_parameter_a)**0.67) / m_bh_solar

    # In T₀C framework: lattice absorbs a fraction, enhancing effective damping
    # The phase-slip acts as an energy drain, reducing the Q-factor
    gamma_lattice = gamma_gr * (1.0 + 2.0 * lattice_damping_factor)

    # Time grid (ringdown lasts ~10 cycles for 65 M_sun BH)
    period_ringdown = 2 * np.pi / omega_qnm_fund
    t_max = num_oscillations * period_ringdown
    t = np.linspace(0, t_max, 2000)

    # Strain amplitudes
    A0 = 1.0  # Normalized initial amplitude
    strain_gr = A0 * np.exp(-gamma_gr * t) * np.cos(omega_qnm_fund * t)
    strain_lattice = A0 * np.exp(-gamma_lattice * t) * np.cos(omega_qnm_fund * t)

    # Energy ratios (integrated power)
    energy_gr = np.cumsum(strain_gr**2) / len(t)
    energy_lattice = np.cumsum(strain_lattice**2) / len(t)
    energy_lost_to_lattice = energy_gr - energy_lattice

    return {
        'time': t,
        'strain_gr': strain_gr,
        'strain_lattice': strain_lattice,
        'omega_qnm': omega_qnm_fund,
        'gamma_gr': gamma_gr,
        'gamma_lattice': gamma_lattice,
        'energy_gr': energy_gr,
        'energy_lattice': energy_lattice,
        'energy_drained_by_lattice': energy_lost_to_lattice,
        'total_drained_fraction': float(energy_lost_to_lattice[-1] / (energy_gr[-1] + 1e-16))
    }

# Run for GW150914 binary (approximately 65 M_sun merger)
ringdown_data = gw_ringdown_with_lattice_damping(
    m_bh_solar=65.0,
    spin_parameter_a=0.7,
    lattice_damping_factor=CONFIG['physics']['r_res_derived_flux_ratio'], # Using derived R_res from CONFIG
    num_oscillations=10
)

logger.info(f"GW Ringdown Analysis (GW150914-like):")
logger.info(f"  QNM frequency: {ringdown_data['omega_qnm']:.4f} rad/s")
logger.info(f"  GR decay rate: {ringdown_data['gamma_gr']:.6f}")
logger.info(f"  Lattice-enhanced decay rate: {ringdown_data['gamma_lattice']:.6f}")
logger.info(f"  Anomalous damping: +{(ringdown_data['gamma_lattice']/ringdown_data['gamma_gr'] - 1)*100:.2f}%")
logger.info(f"  Energy drained to lattice: {ringdown_data['total_drained_fraction']*100:.2f}%")


# Visualization for GW Ringdown
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(ringdown_data['time'], ringdown_data['strain_gr'], 'b-', lw=2, label='GR (standard)', alpha=0.7)
ax.plot(ringdown_data['time'], ringdown_data['strain_lattice'], 'r--', lw=2, label='T₀C (lattice damping)', alpha=0.7)
ax.set_xlabel('Time (M)')
ax.set_ylabel('GW Strain Amplitude')
ax.set_title('Ringdown Damping: GR vs. T₀C Lattice', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.semilogy(ringdown_data['time'], ringdown_data['energy_gr'], 'b-', lw=2, label='GR', alpha=0.7)
ax.semilogy(ringdown_data['time'], ringdown_data['energy_lattice'], 'r--', lw=2, label='T₀C', alpha=0.7)
ax.set_xlabel('Time (M)')
ax.set_ylabel('Integrated Energy (log scale)')
ax.set_title('Energy Decay Comparison', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

ax = axes[1, 0]
ax.fill_between(ringdown_data['time'], 0, ringdown_data['energy_drained_by_lattice'], alpha=0.4, color='orange')
ax.plot(ringdown_data['time'], ringdown_data['energy_drained_by_lattice'], 'orange', lw=2)
ax.set_xlabel('Time (M)')
ax.set_ylabel('Energy Drained to Lattice (W)')
ax.set_title('Phase-Slip Energy Drain', fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
decay_ratio = ringdown_data['gamma_lattice'] / ringdown_data['gamma_gr']
ax.bar(['GR', 'T₀C'], [1.0, decay_ratio], color=['blue', 'red'], alpha=0.6, width=0.4)
ax.axhline(1.0, color='k', ls='--', lw=1, alpha=0.5)
ax.set_ylabel('Relative Decay Rate')
ax.set_title(f'Anomalous Damping Factor: {decay_ratio:.3f}×', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
summary_date = datetime.now().strftime('%Y%m%d_%H%M') # Re-defining if not in scope yet
plt.savefig(f'tzoc_ringdown_damping_v3_{summary_date}.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

logger.info("✅ GW Ringdown Fingerprinting module implemented and visualized.")

In [ ]:
# @title Phase 3: Lattice Visualizations

plt.style.use('dark_background')
fig, axes = plt.subplots(3, 2, figsize=(15, 13))
fig.suptitle("T'Z₀C Lattice Analysis Suite v3 - Core Results", fontsize=16, fontweight='bold')

# Row 1: Basic saturation
axes[0, 0].plot(sat_basic.energy[:500], color='cyan', lw=1.5, label='Energy')
axes[0, 0].axhline(cfg_lat['pc_threshold'], color='r', ls='--', lw=1, label='Threshold')
axes[0, 0].fill_between(range(500), sat_basic.energy[:500], alpha=0.2, color='cyan')
axes[0, 0].set_title('Basic Saturation (drift + noise)', fontweight='bold')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Energy Level')
axes[0, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

safe_loglog(axes[0, 1], freqs_basic[1:], psd_basic[1:], color='lime', lw=1.5)
axes[0, 1].set_title('PSD (Basic)', fontweight='bold')
axes[0, 1].set_xlabel('Frequency (normalized)')
axes[0, 1].set_ylabel('Power')
axes[0, 1].grid(True, which='both', alpha=0.3)

# Row 2: Enhanced saturation + coupling
ax_enh = axes[1, 0]
ax_enh.plot(sat_enhanced.energy[:1000], color='magenta', lw=1.5, label='Energy')
ax_enh_twin = ax_enh.twinx()
ax_enh_twin.plot(sat_enhanced.coupling[:1000], color='yellow', lw=0.8, alpha=0.6, label='Coupling')
ax_enh.axhline(0.92, color='r', ls='--', lw=1, alpha=0.7)
ax_enh.set_title('Enhanced Saturation + Coupling', fontweight='bold')
ax_enh.set_xlabel('Step')
ax_enh.set_ylabel('Energy', color='magenta')
ax_enh_twin.set_ylabel('Coupling Energy', color='yellow')
ax_enh.tick_params(axis='y', labelcolor='magenta')
ax_enh_twin.tick_params(axis='y', labelcolor='yellow')
ax_enh.grid(True, alpha=0.3)

safe_loglog(axes[1, 1], freqs_enhanced[1:], psd_enhanced[1:], color='orange', lw=1.5)
axes[1, 1].set_title('PSD (Enhanced)', fontweight='bold')
axes[1, 1].set_xlabel('Frequency (normalized)')
axes[1, 1].set_ylabel('Power')
axes[1, 1].grid(True, which='both', alpha=0.3)

# Row 3: Resonance sweep + Q-factor effect
ax_res = axes[2, 0]
ax_res.plot(res_df['freq_hz']/1000, res_df['mean_r'], 'o-', color='red', lw=2.5, markersize=6, label='Mean R(t)')
ax_res.axvline(resonance_peak_freq/1000, color='white', ls=':', lw=1.5, alpha=0.7)
ax_res.set_xlabel('Drive Frequency (kHz)', fontweight='bold')
ax_res.set_ylabel('Mean R(t)', color='red', fontweight='bold')
ax_res.tick_params(axis='y', labelcolor='red')
ax_res.set_title(f'Resonance Map (Q≈{cfg_pump["quality_factor"]:.0f})', fontweight='bold')
ax_res.grid(True, ls='--', alpha=0.3)

ax_res_twin = ax_res.twinx()
ax_res_twin.bar(res_df['freq_hz']/1000, res_df['dump_count'], color='yellow', alpha=0.3, width=0.01, label='Dump events')
ax_res_twin.set_ylabel('Dump Events', color='yellow', fontweight='bold')
ax_res_twin.tick_params(axis='y', labelcolor='yellow')

# Row 3, Col 2: Mode distribution (enhanced saturation)
axes[2, 1].fill_between(range(1000), sat_enhanced.mode[:1000], alpha=0.3, color='cyan', label='Mode')
axes[2, 1].set_title('Mode Dynamics (Enhanced)', fontweight='bold')
axes[2, 1].set_xlabel('Step')
axes[2, 1].set_ylabel('Mode State (0/1)')
axes[2, 1].set_ylim(-0.1, 1.1)
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].legend(fontsize=9)

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
summary_date = datetime.now().strftime('%Y%m%d_%H%M')
fig.savefig(f'tzoc_lattice_v3_{summary_date}.png', dpi=150, bbox_inches='tight')
logger.info(f"✅ Saved: tzoc_lattice_v3_{summary_date}.png")

plt.show()
plt.close(fig)


In [ ]:
# @title Phase 3.5: Birefringence Polarization Analysis

logger.info("\n" + "="*60)
logger.info("PHASE 3.5: Gravitational Wave Birefringence Polarization Analysis")
logger.info("="*60)

def gravitational_wave_birefringence(
    gw_strain_plus: np.ndarray,
    gw_strain_cross: np.ndarray,
    lattice_stagger_angle_deg: float = 19.4712,
    propagation_direction: str = 'z'
) -> dict:
    """
    Model birefringence of GW due to discrete tetrahedral lattice.

    The lattice's 19.47° stagger creates a polarization-dependent phase shift.
    + polarization and × polarization experience different effective indices of refraction.

    Args:
        gw_strain_plus, gw_strain_cross: GW polarization components (time series)
        lattice_stagger_angle_deg: Stagger angle of lattice (δ = 19.4712°)
        propagation_direction: Direction of GW propagation ('z', 'x', or arbitrary)

    Returns:
        dict with phase shifts, birefringence parameter, predicted observable effects
    """
    stagger_rad = np.radians(lattice_stagger_angle_deg)

    # Birefringence parameter: lattice-induced refractive index difference
    # Derived from the mismatch between primal and dual angles
    delta_n = np.sin(stagger_rad) / 1000  # Small perturbation regime

    # Propagation distance (assume light-travel distance ~ 1 Gpc for GW150914)
    distance_gpc = 0.4  # 400 Mpc (GW150914 distance)
    distance_meters = distance_gpc * 3.086e25 # 1 parsec = 3.086e16 meters; 1 Gpc = 1e9 pc

    # Phase accumulation due to birefringence
    # Δφ = (π / λ) Δn L, where L is propagation distance
    wavelength_gw = 100 * 1000  # ~100 km for 250 Hz GW (typical LIGO band)
    phase_shift_plus = (np.pi / wavelength_gw) * delta_n * distance_meters
    phase_shift_cross = -phase_shift_plus  # Opposite sign for orthogonal polarization

    # Apply phase shift to strain components
    # Convert to complex for phase shift application
    strain_plus_shifted = gw_strain_plus * np.exp(1j * phase_shift_plus)
    strain_cross_shifted = gw_strain_cross * np.exp(1j * phase_shift_cross)

    # Compute the polarization ellipse parameters (Stokes parameters)
    # Assuming real strain components for initial calculation, then complex for shifted
    # This part can be simplified, as we're interested in the *change*

    # For the purpose of showing effect, we will show the change in relative phase
    # The actual Stokes parameters calculation for a full polarization ellipse is more involved
    # and might require a full complex waveform representation.
    # Simplified observable: degree of circular polarization (DCP)
    # Which is related to V (circular polarization Stokes parameter)

    # Let's use the instantaneous amplitudes and phases from Hilbert transform for a more accurate DCP
    analytic_plus = hilbert(gw_strain_plus)
    analytic_cross = hilbert(gw_strain_cross)

    # Instantaneous phase difference
    inst_phase_plus = np.unwrap(np.angle(analytic_plus))
    inst_phase_cross = np.unwrap(np.angle(analytic_cross))

    # Phase difference before and after birefringence
    phase_diff_original = (inst_phase_plus - inst_phase_cross) % (2 * np.pi)
    phase_diff_shifted = (inst_phase_plus + phase_shift_plus - (inst_phase_cross + phase_shift_cross)) % (2 * np.pi)

    # Degree of circular polarization (simplified concept for demonstration)
    # DCP is typically related to the ellipticity and orientation of the polarization ellipse
    # For linear GWs, DCP is ideally 0. A non-zero DCP indicates circular polarization.
    # A rough estimate for change due to birefringence might be proportional to sin(phase_shift)

    # This is a conceptual representation of how birefringence would manifest
    dcp = np.abs(np.sin(phase_shift_plus)) * np.abs(gw_strain_plus) # Simplified for demonstration

    # Poincaré sphere rotation is a rotation of the polarization ellipse
    # A single number for rotation might not be representative for a time series
    # We'll return the phase shift itself as the most direct observable

    return {
        'delta_n': delta_n,
        'phase_shift_plus_rad': phase_shift_plus,
        'phase_shift_cross_rad': phase_shift_cross,
        'strain_plus_shifted': strain_plus_shifted,
        'strain_cross_shifted': strain_cross_shifted,
        'degree_circular_polarization': dcp,
        'poincare_rotation_deg': np.degrees(phase_shift_plus - phase_shift_cross), # Total change in relative phase
        'birefringence_observable': True if np.max(dcp) > 1e-6 else False # Check for small but non-zero effect
    }

# Test with synthetic GW150914 strain
# Create realistic chirp signal
t_chirp = np.linspace(0, 0.2, 4000)  # 0.2 s at 20 kHz sampling
f_sweep_start, f_sweep_end = 35, 250
chirp_rate = (f_sweep_end - f_sweep_start) / 0.2
inst_freq = f_sweep_start + chirp_rate * t_chirp
phase_chirp = 2 * np.pi * (f_sweep_start * t_chirp + 0.5 * chirp_rate * t_chirp**2)
amplitude_envelope = np.exp(-10 * t_chirp)  # Taper off post-merger

strain_plus_synthetic = amplitude_envelope * np.sin(phase_chirp)
strain_cross_synthetic = amplitude_envelope * 0.7 * np.cos(phase_chirp)  # Detector angle ~45°

biref_result = gravitational_wave_birefringence(
    strain_plus_synthetic, strain_cross_synthetic,
    lattice_stagger_angle_deg=19.4712
)

logger.info(f"GW Birefringence Analysis:")
logger.info(f"  Lattice-induced Δn: {biref_result['delta_n']:.2e}")
logger.info(f"  Phase shift (+ pol): {biref_result['phase_shift_plus_rad']:.6f} rad")
logger.info(f"  Max DCP (circular polarization): {np.max(biref_result['degree_circular_polarization']):.6f}")
logger.info(f"  Poincaré rotation equivalent: {np.mean(biref_result['poincare_rotation_deg']):.3f}°")
logger.info(f"  Observable effect: {biref_result['birefringence_observable']}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Polarization states before and after
ax = axes[0, 0]
# Using real part of complex strains for visualization of 'waveform'
ax.plot(t_chirp, np.real(strain_plus_synthetic), 'b-', lw=1, alpha=0.7, label='Original Strain+')
ax.plot(t_chirp, np.real(biref_result['strain_plus_shifted']), 'r--', lw=1.5, alpha=0.7, label='Lattice-shifted Strain+')
ax.set_xlabel('Time')
ax.set_ylabel('Strain+')
ax.set_title('Strain+ Polarization: Original vs. Lattice-Shifted', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# DCP evolution
ax = axes[0, 1]
ax.plot(t_chirp, biref_result['degree_circular_polarization'], 'orange', lw=2)
ax.axhline(0, color='k', ls='--', lw=0.5)
ax.fill_between(t_chirp, 0, biref_result['degree_circular_polarization'], alpha=0.3, color='orange')
ax.set_xlabel('Time')
ax.set_ylabel('Degree of Circular Polarization')
ax.set_title('Anomalous Circular Polarization (Lattice Effect)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Phase shift accumulation
ax = axes[1, 0]
ax.text(0.5, 0.5, f"Phase Shift (+ pol):\n{biref_result['phase_shift_plus_rad']:.4f} rad\n({np.degrees(biref_result['phase_shift_plus_rad']):.3f}°)",
        ha='center', va='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.axis('off')
ax.set_title('Quantified Phase Shift', fontweight='bold')

# Poincaré rotation
ax = axes[1, 1]
mean_poincare_rotation = np.mean(biref_result['poincare_rotation_deg']) # Use mean as it's constant if shifts are constant
ax.bar(['Original GW', 'GW + T₀C lattice'], [0, mean_poincare_rotation], color=['blue', 'red'], alpha=0.6)
ax.set_ylabel('Poincaré Rotation (°)')
ax.set_title('Observable Polarization Rotation', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
summary_date = datetime.now().strftime('%Y%m%d_%H%M') # Ensure summary_date is defined for plot saving
plt.savefig(f'tzoc_birefringence_v3_{summary_date}.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

logger.info("✅ GW Birefringence module implemented and visualized.")

In [ ]:
# @title Phase 4: Synthetic Bench Test Data (Validation)

logger.info("\n" + "="*60)
logger.info("PHASE 4: Synthetic Bench Test & Model Validation")
logger.info("="*60)

# Generate synthetic "observed" velocity dispersion profile
# Mimics observed galaxy cluster velocity measurements
def synthetic_velocity_profile(r_norm: np.ndarray, center_vel: float = 800.0,
                               scale_length: float = 0.3, noise_level: float = 50.0) -> np.ndarray:
    """
    Synthetic velocity dispersion profile: Gaussian core + power-law decline.
    Realistic for galaxy clusters (Coma, Virgo-like).
    """
    # Hernquist-like profile: ν(r) = ν₀ / (1 + r/r_s)^0.5
    sigma_r = center_vel / (1.0 + (r_norm / scale_length)**2)**0.25
    # Add realistic measurement noise
    sigma_r += np.random.normal(0, noise_level, len(r_norm))
    return np.maximum(sigma_r, 50.0)  # Minimum physical velocity

# Create bench test dataset
r_bench_norm = np.linspace(0, 1.5, 20)  # Normalized radial distance
v_obs_bench = synthetic_velocity_profile(r_bench_norm, center_vel=900.0, scale_length=0.4, noise_level=60.0)

# Model prediction from resonance sweep
# Map mean_r (phase coherence) to velocity via effective gravitational gradient
r_model_norm = np.linspace(0, 1.5, len(res_df))
# Invert the resonance curve: lower R → lower velocity dispersion
v_pred_bench = 200.0 + 700.0 * np.interp(r_model_norm,
                                          [0, len(res_df)//2, len(res_df)-1],
                                          [res_df['mean_r'].iloc[-1],
                                           res_df['mean_r'].iloc[len(res_df)//2],
                                           res_df['mean_r'].iloc[0]])

# Compute goodness-of-fit
gof = compute_gof(v_pred_bench, v_obs_bench[:len(v_pred_bench)])
logger.info(f"Bench Test Model-Observation Comparison:")
logger.info(f"  R² = {gof['r_squared']:.4f}")
logger.info(f"  RMSE = {gof['rmse']:.1f} km/s")
logger.info(f"  Correlation = {gof['correlation']:.4f}")

# Bench test visualization
fig_bench, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig_bench.suptitle('Bench Test: Model vs. Synthetic Observations', fontsize=14, fontweight='bold')

# Panel 1: Velocity profiles
ax1.plot(r_bench_norm, v_obs_bench, 'o-', color='red', lw=2.5, markersize=7, label='Synthetic Obs', alpha=0.8)
ax1.plot(r_model_norm, v_pred_bench, 's--', color='cyan', lw=2.5, markersize=6, label='Model Pred', alpha=0.8)
ax1.fill_between(r_bench_norm, v_obs_bench - 100, v_obs_bench + 100, alpha=0.15, color='red')
ax1.set_xlabel('Normalized Radius', fontweight='bold')
ax1.set_ylabel('Velocity Dispersion (km/s)', fontweight='bold')
ax1.set_title('Profile Comparison', fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Panel 2: Residuals + GOF metrics
residuals = v_pred_bench - v_obs_bench[:len(v_pred_bench)]
ax2.bar(r_model_norm, residuals, color='yellow', alpha=0.6, width=0.05)
ax2.axhline(0, color='white', ls='--', lw=1)
ax2.set_xlabel('Normalized Radius', fontweight='bold')
ax2.set_ylabel('Residual (km/s)', fontweight='bold')
ax2.set_title(f'Residuals (R²={gof["r_squared"]:.3f}, RMSE={gof["rmse"]:.1f})', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
bench_date = datetime.now().strftime('%Y%m%d_%H%M')
fig_bench.savefig(f'tzoc_bench_test_v3_{bench_date}.png', dpi=150, bbox_inches='tight')
logger.info(f"✅ Saved: tzoc_bench_test_v3_{bench_date}.png")
plt.show()
plt.close(fig_bench)

In [ ]:
# @title Phase 4.5: Phase-Slip Hysteresis & Memory Effect

logger.info("\n" + "="*60)
logger.info("PHASE 4.5: Phase-Slip Hysteresis & Memory Effect")
logger.info("="*60)

def phase_slip_hysteresis(
    drive_signal: np.ndarray,
    time_array: np.ndarray,
    memory_decay_time: float = 0.5,
    stagger_angle_deg: float = 19.4712
) -> dict:
    """
    Model the phase-slip handshake with memory (hysteresis).

    The lattice does not instantaneously reset; it "remembers" its prior state
    through a stored tension field. This creates path-dependent behavior.

    Args:
        drive_signal: Input driving force (time series)
        time_array: Corresponding time points
        memory_decay_time: Timescale over which spatial tension relaxes
        stagger_angle_deg: Lattice stagger angle (sets phase-slip profile)

    Returns:
        dict with: tension_field, phase_slip_envelope, work_done, hysteresis_loss
    """
    stagger_rad = np.radians(stagger_angle_deg)
    dt = time_array[1] - time_array[0]
    n_steps = len(time_array)

    # Memory kernel: exponential relaxation
    tau_mem = memory_decay_time

    # Initialize fields
    tension_field = np.zeros(n_steps)  # Spatial tension (accumulated state)
    phase_slip = np.zeros(n_steps)     # Phase-slip envelope
    work_accumulated = np.zeros(n_steps)  # Work done against lattice resistance

    # Iterate forward in time with memory
    for i in range(1, n_steps):
        # Tension relaxes exponentially
        tension_field[i] = tension_field[i-1] * np.exp(-dt / tau_mem)

        # Drive adds to tension (modulo the stagger gate)
        tension_input = drive_signal[i] * np.cos(stagger_rad)
        tension_field[i] += tension_input * dt

        # Phase-slip is triggered when tension exceeds threshold
        slip_threshold = 0.5
        if tension_field[i] > slip_threshold:
            phase_slip[i] = tension_field[i] * np.sin(stagger_rad)
            # Slip releases tension proportionally
            tension_field[i] *= (1.0 - 0.7 * np.sin(stagger_rad))

        # Work = ∫ F · dx (dissipated to lattice memory)
        work_accumulated[i] = work_accumulated[i-1] + drive_signal[i] * phase_slip[i] * dt

    # Hysteresis loss: energy that could have propagated freely but was stored/dissipated
    ideal_work = np.sum(drive_signal * np.abs(drive_signal) * dt) # Proxy for ideal work without loss
    hysteresis_loss = ideal_work - work_accumulated[-1]

    return {
        'time': time_array,
        'tension_field': tension_field,
        'phase_slip': phase_slip,
        'work_accumulated': work_accumulated,
        'hysteresis_loss': hysteresis_loss,
        'ideal_work': ideal_work,
        'efficiency': (work_accumulated[-1] / (ideal_work + 1e-16)) if ideal_work > 0 else 0.0
    }

# Generate a multi-cycle driving signal (like LIGO driving a cavity)
dt_drive = 1e-4
t_drive = np.arange(0, 10, dt_drive)  # 10 seconds of driving
drive_freq = 250  # Hz (typical LIGO band)
drive_amp = 0.1
drive_signal = drive_amp * np.sin(2 * np.pi * drive_freq * t_drive)

hysteresis_result = phase_slip_hysteresis(
    drive_signal, t_drive,
    memory_decay_time=0.5,
    stagger_angle_deg=19.4712
)

logger.info(f"Phase-Slip Hysteresis Analysis:")
logger.info(f"  Total work (ideal): {hysteresis_result['ideal_work']:.6e}")
logger.info(f"  Accumulated work: {hysteresis_result['work_accumulated'][-1]:.6e}")
logger.info(f"  Hysteresis loss: {hysteresis_result['hysteresis_loss']:.6e}")
logger.info(f"  Transmission efficiency: {hysteresis_result['efficiency']*100:.2f}%")
logger.info(f"  Memory-induced attenuation: {(1 - hysteresis_result['efficiency'])*100:.2f}%")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tension field evolution
ax = axes[0, 0]
ax.plot(hysteresis_result['time'][:2000], hysteresis_result['tension_field'][:2000], 'b-', lw=1.5)
ax.fill_between(hysteresis_result['time'][:2000], hysteresis_result['tension_field'][:2000], alpha=0.3)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Spatial Tension')
ax.set_title('Lattice Tension Field (Memory Effect)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Phase-slip envelope
ax = axes[0, 1]
ax.plot(hysteresis_result['time'][:2000], hysteresis_result['phase_slip'][:2000], 'r-', lw=1.5)
ax.fill_between(hysteresis_result['time'][:2000], hysteresis_result['phase_slip'][:2000], alpha=0.3, color='red')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Phase-Slip Magnitude')
ax.set_title('Phase-Slip Events (Handshake Sequence)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Work accumulation (hysteresis loop)
ax = axes[1, 0]
ax.plot(hysteresis_result['work_accumulated'][:2000], 'g-', lw=2, label='Accumulated')

# To correctly plot the ideal work line, it should span the same length as work_accumulated
ideal_work_per_step = hysteresis_result['ideal_work'] / len(hysteresis_result['work_accumulated']) if len(hysteresis_result['work_accumulated']) > 0 else 0
ideal_work_cumulative = np.cumsum([ideal_work_per_step] * len(hysteresis_result['work_accumulated']))
ax.plot(ideal_work_cumulative[:2000], 'k--', lw=1.5, label='Ideal (no loss)')

ax.fill_between(range(2000), hysteresis_result['work_accumulated'][:2000],
                ideal_work_cumulative[:2000],
                alpha=0.3, color='yellow', label='Hysteresis loss')
ax.set_xlabel('Time index')
ax.set_ylabel('Cumulative Work (J)')
ax.set_title('Hysteresis Loop (Energy Storage)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Efficiency metric
ax = axes[1, 1]
ax.bar(['Ideal (no lattice)', 'T₀C (with lattice)'], [100, hysteresis_result['efficiency']*100],
       color=['blue', 'red'], alpha=0.6, width=0.4)
ax.set_ylabel('Energy Transmission Efficiency (%)')
ax.set_title('Memory-Induced Attenuation', fontweight='bold')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
summary_date = datetime.now().strftime('%Y%m%d_%H%M')
plt.savefig(f'tzoc_hysteresis_v3_{summary_date}.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

logger.info("✅ Phase-Slip Hysteresis & Memory Effect module implemented and visualized.")

In [ ]:
# @title Phase 5: Summary Report & Export (Consolidated)

logger.info("\n" + "="*60)
logger.info("SUMMARY & COMPREHENSIVE EXPORT")
logger.info("="*60)

summary_date_iso = datetime.now().date().isoformat()
summary_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Consolidated summary
summary_data = {
    'metadata': {
        'date': summary_date_iso,
        'time': summary_time,
        'version': '3.0-revised',
        'seed': CONFIG['seed'],
        'config_snapshot': CONFIG # Include full config for reproducibility
    },
    'lattice_saturation': {
        'basic_resets': sat_basic.reset_count,
        'basic_mean_energy': sat_basic.mean_energy,
        'basic_peak_energy': sat_basic.peak_energy,
        'enhanced_resets': sat_enhanced.reset_count,
        'enhanced_mean_energy': sat_enhanced.mean_energy,
        'enhanced_peak_energy': sat_enhanced.peak_energy,
    },
    'wave_pump': {
        'sample_mean_r': pump_sample.mean_r,
        'sample_std_r': pump_sample.std_r,
        'sample_peak_r': pump_sample.peak_r,
        'sample_dump_events': pump_sample.dump_events,
        'sample_energy_dissipated': pump_sample.energy_dissipated,
        'quality_factor': CONFIG['wave_pump']['quality_factor'],
    },
    'resonance_sweep': {
        'num_frequencies': len(res_df),
        'peak_frequency_hz': resonance_peak_freq,
        'peak_mean_r': float(res_df.loc[resonance_peak_idx, 'mean_r']),
        'frequency_range_hz': [float(res_df['freq_hz'].min()), float(res_df['freq_hz'].max())],
    },
    'bench_test': {
        'model_r_squared': gof['r_squared'],
        'model_rmse_km_s': gof['rmse'],
        'model_correlation': gof['correlation'],
        'synthetic_obs_count': len(v_obs_bench),
        'synthetic_obs_mean': float(np.mean(v_obs_bench)),
        'synthetic_obs_std': float(np.std(v_obs_bench)),
    },
    'physics_constants': {
        'alpha_physical': alpha_phys,
        'alpha_model': model_alpha,
        'alpha_gap': gap,
        'r_res_derived_flux_ratio': float(CONFIG['physics']['r_res_derived_flux_ratio']) # Get derived R_res from CONFIG
    }
}

# Flatten for CSV export
def flatten_dict(d, parent_key='', sep=''):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep='_').items())
        else:
            items.append((new_key, v))
    return dict(items)

flat_summary = flatten_dict(summary_data)

# Print summary
logger.info("\n--- SUMMARY TABLE ---")
for key, val in flat_summary.items():
    logger.info(f"{key}: {val}")

# Export to CSV (existing functionality)
csv_filename = f'tzoc_summary_v3_{summary_date_iso}.csv'
with open(csv_filename, 'w') as f:
    f.write('parameter,value\n')
    for key, val in flat_summary.items():
        f.write(f'{key},{val}\n')
logger.info(f"✅ Exported summary: {csv_filename}")

# Export resonance sweep data
res_export = res_df.copy()
res_export['freq_khz'] = res_export['freq_hz'] / 1000
res_filename = f'tzoc_resonance_sweep_v3_{summary_date_iso}.csv'
res_export.to_csv(res_filename, index=False)
logger.info(f"✅ Exported resonance data: {res_filename}")

# Export bench test data
bench_export = pd.DataFrame({
    'radius_normalized': r_model_norm,
    'velocity_pred_km_s': v_pred_bench,
    'velocity_obs_km_s': v_obs_bench[:len(v_pred_bench)],
    'residual_km_s': residuals,
    'model_r': res_df['mean_r'].values,
})
bench_filename = f'tzoc_bench_test_v3_{summary_date_iso}.csv'
bench_export.to_csv(bench_filename, index=False)
logger.info(f"✅ Exported bench test data: {bench_filename}")

# Export saturation energies separately to handle different lengths
sat_basic_export = pd.DataFrame({
    'step': range(len(sat_basic.energy)),
    'energy': sat_basic.energy,
})
sat_basic_filename = f'tzoc_saturation_basic_v3_{summary_date_iso}.csv'
sat_basic_export.to_csv(sat_basic_filename, index=False)
logger.info(f"✅ Exported basic saturation dynamics: {sat_basic_filename}")

sat_enhanced_export = pd.DataFrame({
    'step': range(len(sat_enhanced.energy)),
    'energy': sat_enhanced.energy,
    'coupling': sat_enhanced.coupling,
    'mode': sat_enhanced.mode,
})
sat_enhanced_filename = f'tzoc_saturation_enhanced_v3_{summary_date_iso}.csv'
sat_enhanced_export.to_csv(sat_enhanced_filename, index=False)
logger.info(f"✅ Exported enhanced saturation dynamics: {sat_enhanced_filename}")


# ===================== HDF5 EXPORT (Consolidated) =====================
def export_to_hdf5(summary_data: dict, resonance_df: pd.DataFrame, bench_df: pd.DataFrame,
                   sat_basic_df: pd.DataFrame, sat_enhanced_df: pd.DataFrame, filename: str):
    """Exports all simulation results to a single HDF5 file."""
    try:
        import h5py
        with h5py.File(filename, 'w') as f:
            # Store metadata as JSON string attribute
            f.attrs['summary_json'] = json.dumps(summary_data, indent=2)

            # Store dataframes as separate HDF5 datasets
            f.create_group('dataframes')
            f['dataframes'].create_dataset('resonance_sweep', data=resonance_df.to_records(index=False))
            f['dataframes'].create_dataset('bench_test', data=bench_df.to_records(index=False))
            f['dataframes'].create_dataset('saturation_basic', data=sat_basic_df.to_records(index=False))
            f['dataframes'].create_dataset('saturation_enhanced', data=sat_enhanced_df.to_records(index=False))

        logger.info(f"✅ Exported consolidated results to HDF5: {filename}")
    except ImportError:
        logger.warning("H5py not installed. Skipping HDF5 export. Install with 'pip install h5py' for this feature.")
    except Exception as e:
        logger.error(f"Error exporting to HDF5: {e}")

hdf5_filename = f'tzoc_consolidated_v3_{summary_date_iso}.hdf5'
export_to_hdf5(summary_data, res_export, bench_export, sat_basic_export, sat_enhanced_export, hdf5_filename)

logger.info("\n" + "="*60)
logger.info(f"Analysis complete: {summary_time}")
logger.info("="*60)

## Re-orienting the Framework: From Global Bounds to Local Lattice Mechanics

The previous interpretation of the framework's predictions regarding gravitational wave power ceilings has been critiqued for its ad-hoc global scaling. This section re-orients the framework towards a more robust, **truth-seeking version** that focuses on **local, microscopic lattice constraints** and generates **falsifiable modifications to gravitational wave characteristics** rather than speculative global bounds.

---

### 1. Correcting the Scale Mistake: Global vs. Local

The fundamental error was attempting to directly scale the global $c^5/G$ luminosity bound. This bound is an asymptotic limit derived from macroscopic horizon geometry. In contrast, the tetrahedral phase-sync slip (related to $\alpha$) and the $19.47^\circ$ stagger are **local, microscopic property constraints of the vacuum lattice**.

Instead of claiming a hard global power ceiling for entire macroscopic systems, the framework now posits: **The phase-slip sets the maximum local energy-flux density (Watts/m²) that a single tetrahedral voxel can transmit before local topological deformation occurs.** When a massive merger occurs, the total power emitted spans a large macroscopic region. A larger system can naturally emit more total power ($10^{51}\text{ W}$ or higher) simply because more spatial voxels are participating in parallel. The true geometric bottleneck is local, not global.

---

### 2. The Predictive Path Forward: Lattice-Driven Waveform Deviations

To generate clean, falsifiable predictions that next-generation detectors (LIGO O5, LISA, Einstein Telescope) can actually test, we must look for how a discrete, close-packed tetrahedral lattice alters the *propagation* and *ringdown* of gravitational waves.

Instead of an arbitrary power cap, the framework predicts specific, subtle deviations from General Relativity during the most extreme, high-flux moments of a merger:

#### A. Modified Ringdown Damping (The Phase-Slip Drain)

When a newly merged black hole relaxes (the "ringdown" phase), it vibrates, emitting gravitational waves until it settles into a stable sphere. In standard GR, this damping is dictated purely by the black hole's mass and spin.

In the Moving Space Framework, the high-frequency geometric shear forces the local space matrix to oscillate through the $19.47^\circ$ stagger gate. Because the phase-sync coupling (related to $\alpha$) has a finite transmission tolerance, a predictable fraction of the wave's energy will experience an impedance mismatch, leaking into the non-interactive residue channel ($R_{\text{res}}$).

*   **Falsifiable Prediction:** The ringdown phases of highly luminous mergers will exhibit an anomalous damping rate—a "fractional energy drain"—that deviates from standard GR templates. This deviation will scale deterministically with the local shear intensity, bounded by $\alpha$ and $R_{\text{res}}$.

#### B. Gravitational Wave Birefringence (Tetrahedral Projections)

Because the background matrix is structured around an $sp^3$ tetrahedral basis ($\theta_+ = 109.47^\circ, \theta_- = 70.53^\circ$) rather than a flat, isotropic continuum, gravitational waves traveling across vast cosmic distances must project through these discrete spatial angles.

This introduces a subtle, frequency-dependent **birefringence** or polarization shift. As a gravitational wave propagates:

*   The "Plus" ($+$) and "Cross" ($\times$) polarization modes will experience minutely different propagation velocities or phase shifts depending on their alignment relative to the underlying tetrahedral axes of the local volume.
*   **Falsifiable Prediction:** By analyzing the polarization correlation of cosmic events across widely separated detectors, next-gen systems should detect a subtle, periodic angular dependence in the wave's polarization strain that matches the tetrahedral symmetry factor ($\frac{\delta}{\theta_+ - \theta_-} = 0.5$).

---

### 3. Refining the Target for $R_{\text{res}}$

Following this cleaner path, the residue factor $R_{\text{res}} \approx 0.1291$ found in the high-precision simulation is no longer an unknown global loss. It is the **local vacuum polarization cutoff**.

It represents the exact ratio of energy that successfully cross-encodes between a linear, propagating wave (**Straight-Mode**) and a localized quantum lattice excitation (**Loop-Mode**). By embedding $R_{\text{res}}$ directly into the local coupling equations of the vacuum, it serves as the foundational parameter for predicting the point where smooth spacetime routing breaks down into discrete quantum states—acting as a natural Planck-scale regulator without requiring infinite mathematical singularities.

## Optional: SDSS & LIGO Integration (Robust Stubs)

The cells below are provided as **stubs** for when real astronomical data is available.
They maintain the same architecture as the bench test but gate access behind availability checks.

In [ ]:
# @title [Optional] SDSS Real Data Integration

SDSS_ENABLED = False  # Set to True if astroquery is available

if SDSS_ENABLED:
    try:
        from astroquery.sdss import SDSS
        from astropy import coordinates as coords
        import astropy.units as u

        cfg_sdss = CONFIG['sdss']
        pos = coords.SkyCoord(ra=cfg_sdss['ra'], dec=cfg_sdss['dec'], unit='deg')

        # Photometric query
        query = f"""
        SELECT ra, dec, modelMag_r FROM PhotoObjAll
        WHERE mode = 1 AND type = 6
          AND ra BETWEEN {pos.ra.deg - 1} AND {pos.ra.deg + 1}
          AND dec BETWEEN {pos.dec.deg - 0.5} AND {pos.dec.deg + 0.5}
        LIMIT 10000
        """
        sdss_data = SDSS.query_sql(query, data_release=12)
        logger.info(f"SDSS: Fetched {len(sdss_data)} photometric objects")

        # Spectroscopic sample
        spec_data = SDSS.query_region(pos, radius=0.1*u.deg, spectro=True)
        if spec_data and len(spec_data) > 0:
            z_vals = spec_data['z'].value
            v_los = z_vals * 299792.458
            logger.info(f"SDSS: {len(spec_data)} spectra, <v_los> = {np.mean(v_los):.0f} km/s")
    except Exception as e:
        logger.warning(f"SDSS integration failed (expected if offline): {e}")
else:
    logger.info("SDSS integration disabled (set SDSS_ENABLED=True to activate)")


In [ ]:
# @title [Optional] LIGO Real Data Integration

LIGO_ENABLED = True  # Set to True if gwpy is available

if LIGO_ENABLED:
    try:
        from gwpy.timeseries import TimeSeries

        cfg_ligo = CONFIG['ligo']
        data = TimeSeries.fetch_open_data('H1', cfg_ligo['gw_start_time'],
                                          cfg_ligo['gw_start_time'] + 32)

        white = data.whiten()
        bp = white.bandpass(cfg_ligo['bandpass_low'], cfg_ligo['bandpass_high'])

        # Phase analysis on chirp window
        t_merge = cfg_ligo['gw_start_time'] + cfg_ligo['merger_time_offset']
        chirp_win = bp.crop(t_merge - cfg_ligo['chirp_duration']/2,
                            t_merge + cfg_ligo['chirp_duration']/2)

        analytic = hilbert(chirp_win.value)
        inst_phase = np.unwrap(np.angle(analytic))
        phase_rate = np.gradient(inst_phase, chirp_win.times.value)

        logger.info(f"LIGO {cfg_ligo['event_name']}:")
        logger.info(f"  Chirp SNR ≈ {np.std(phase_rate):.2e}")
    except Exception as e:
        logger.warning(f"LIGO integration failed (expected if offline): {e}")
else:
    logger.info("LIGO integration disabled (set LIGO_ENABLED=True to activate)")

## Consolidated Physics and Observational Predictions

This section consolidates key visualizations and numerical summaries from the high-precision tetrahedral integration, gravitational wave ringdown damping, detector sensitivity, and conceptual SNR analyses. It provides a unified view of the T'Z₀C framework's predictions regarding lattice-induced effects on gravitational waves and their potential observability.

In [ ]:
# @title Phase 5.5: Detector Sensitivity & SNR Curves
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Assuming these variables are available from previous cell executions:
# ringdown_data (from 1238bd2b)
# t0c_data (from YPvWv1_BzO0K)
# freqs_snr_filtered, asd_ligo, asd_lisa, asd_et (from d1d233d6)
# spectral_snr_ligo, spectral_snr_lisa, spectral_snr_et (from d1d233d6)
# snr_ligo, snr_lisa, snr_et (from d1d233d6)
# decay_ratio (from 1238bd2b)

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("T'Z₀C Consolidated Physics & Observational Predictions", fontsize=18, fontweight='bold', y=1.02)

# --- Panel 1: GW Ringdown Damping (from YPvWv1_BzO0K and 1238bd2b) ---
ax = axes[0, 0]
# Use data from the more comprehensive ringdown_data from 1238bd2b
ax.plot(ringdown_data['time'], ringdown_data['strain_gr'], 'b-', lw=2, label='GR (standard)', alpha=0.7)
ax.plot(ringdown_data['time'], ringdown_data['strain_lattice'], 'r--', lw=2, label='T₀C (lattice damping)', alpha=0.7)
ax.set_xlabel('Time (M)')
ax.set_ylabel('GW Strain Amplitude')
ax.set_title(f"Ringdown Damping: GR vs. T₀C Lattice (Anomalous Damping: {ringdown_data['total_drained_fraction']*100:.2f}%)", fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel 2: Detector Sensitivity Curves (from d1d233d6) ---
ax = axes[0, 1]
ax.loglog(freqs_snr_filtered, asd_ligo, label='LIGO (Design)', color='cyan', lw=2)
ax.loglog(freqs_snr_filtered, asd_lisa, label='LISA (Conceptual)', color='magenta', lw=2, linestyle='--')
ax.loglog(freqs_snr_filtered, asd_et, label='Einstein Telescope (Conceptual)', color='lime', lw=2, linestyle=':')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude Spectral Density (Hz$^{-1/2}$)')
ax.set_title('Gravitational Wave Detector Sensitivity Curves', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(1e-25, 1e-18) # Consistent y-limit

# --- Panel 3: Conceptual Signal-to-Noise Ratio (SNR) Spectrum (from d1d233d6) ---
ax = axes[1, 0]
ax.semilogx(freqs_snr_filtered, spectral_snr_ligo, label=f'LIGO (SNR={snr_ligo:.1f})', color='cyan', lw=2)
ax.semilogx(freqs_snr_filtered, spectral_snr_lisa, label=f'LISA (SNR={snr_lisa:.1f})', color='magenta', lw=2, linestyle='--')
ax.semilogx(freqs_snr_filtered, spectral_snr_et, label=f'ET (SNR={snr_et:.1f})', color='lime', lw=2, linestyle=':')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Spectral SNR Density (Conceptual)')
ax.set_title('Conceptual Signal-to-Noise Ratio (SNR) Spectrum', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(bottom=0)

# --- Panel 4: Key Numerical Results Summary (Text) ---
ax = axes[1, 1]
ax.axis('off') # Turn off axis for text display

# Use R_res from t0c_data and decay_ratio from ringdown_data
R_res_val = t0c_data["residue_factor_Rres"]
# decay_ratio_val is already calculated in 1238bd2b

summary_text = (
    "Key T'Z₀C Framework Parameters:\n\n"
    f"  Residue Factor (R_res): {R_res_val:.6f}\n"
    f"  GR Decay Rate: {ringdown_data['gamma_gr']:.6f}\n"
    f"  T₀C Decay Rate: {ringdown_data['gamma_lattice']:.6f}\n"
    f"  Anomalous Damping Factor: {decay_ratio:.3f}x\n\n"
    "Predicted Observability (conceptual):\n"
    f"  LIGO SNR: {snr_ligo:.1f}\n"
    f"  LISA SNR: {snr_lisa:.1f}\n"
    f"  ET SNR: {snr_et:.1f}"
)

ax.text(0.05, 0.95, summary_text,
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', fc='gray', alpha=0.2, ec='white'))

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
summary_date = datetime.now().strftime('%Y%m%d_%H%M')
plt.savefig(f'tzoc_consolidated_physics_v3_{summary_date}.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Phase Geometry Verification Summary

This section summarizes the key findings from the `run_phase_geometry_simulation`:

*   **Golden Angle:** The simulation uses a target Golden Angle of approximately `137.5078°` for phase accumulation.
*   **Target Physical Inverse Fine-Structure Constant ($\alpha^{-1}$):** This value is set to approximately `137.0360`.
*   **Thomas Precession Factor:** A factor of `0.5` is integrated into the analysis, stemming from the T'Z₀C geometric framework and its connection to relativistic corrections (Wigner rotation).
*   **Comparison:** Applying the Thomas Precession Factor directly to the Golden Angle (`137.5078° * 0.5 = 68.7539°`) actually *decreases* the fidelity of the approximation to the physical inverse fine-structure constant (from `99.66%` to `50.17%`). This highlights that while the Thomas Factor is a crucial relativistic correction for coupling coefficients, its direct scalar application here does not improve the approximation of $\alpha^{-1}$ itself, aligning with the theoretical documents' emphasis on its role in fixing a factor of 2 in coupling, not in the fundamental derivation of $\alpha$.

In [ ]:
# @title
import math
import numpy as np
from scipy.constants import physical_constants, c, G, alpha as fine_structure_alpha
import json
from datetime import datetime
import matplotlib.pyplot as plt

def run_high_precision_routing() -> dict:
    """
    Executes the Moving Space Framework integration using high-precision physical constants
    and precise tetrahedral matrix angles. Exports data to JSON and returns the raw results dictionary.
    """
    print("=== Moving Space Framework Simulator (High-Precision Tetrahedral Integration) ===")
    print(f"Run at: {datetime.now().isoformat()}\n")

    # 1. High-Precision Physical Constants
    c_val = c
    G_val = G
    alpha = physical_constants['fine-structure constant'][0]

    print("1. PHYSICAL CONSTANTS")
    print(f"c = {c_val:.10e} m/s")
    print(f"G = {G_val:.10e} m³ kg⁻¹ s⁻²")
    print(f"α = {alpha:.15f} (~1/137.035999)")
    print("-" * 80)

    # 2. Ultra-Precise Tetrahedral Geometry (T0C Registry Precision)
    theta_plus_deg = 109.47122063449069  # θ₊
    theta_minus_deg = 70.52877936550931  # θ₋
    delta_deg = 19.47122063449069        # δ (stagger)

    theta_plus = math.radians(theta_plus_deg)
    theta_minus = math.radians(theta_minus_deg)
    delta = math.radians(delta_deg)

    angular_divergence = theta_plus - theta_minus
    geometric_factor = delta / angular_divergence  # Collapses to exactly 0.5

    print("2. TETRAHEDRAL GEOMETRY (T0C Registry Precision)")
    print(f"θ₊ (Primal) = {theta_plus_deg:.10f}°")
    print(f"θ₋ (Dual)   = {theta_minus_deg:.10f}°")
    print(f"δ (Stagger) = {delta_deg:.10f}°")
    print(f"Divergence (θ₊ - θ₋) = {math.degrees(angular_divergence):.10f}°")
    print(f"Geometric factor (δ / 2δ) = {geometric_factor:.12f}  ← Exactly 1/2")
    print("-" * 80)

    # 3. Macro Stiffness (GR)
    macro_stiffness = (c_val**4) / (8.0 * math.pi * G_val)
    print("3. MACRO SPACETIME STIFFNESS")
    print(f"c⁴ / (8πG) = {macro_stiffness:.8e}")
    print("-" * 80)

    # 4. Ideal Geometric Routing
    ideal_routed = geometric_factor * macro_stiffness * alpha
    log10_ideal = math.log10(ideal_routed)

    print("4. IDEAL GEOMETRIC ROUTING")
    print(f"Ideal term (½ × c⁴/8πG × α) = {ideal_routed:.8e}")
    print(f"Order of magnitude: 10^{int(log10_ideal)}")
    print(f"Remainder: 10^{log10_ideal - int(log10_ideal):.4f}")
    print("-" * 80)

    # 5. Real-World Cross-Checks & Residue
    m_p = physical_constants['proton mass'][0]
    m_e = physical_constants['electron mass'][0]
    e = physical_constants['elementary charge'][0]
    k_e = 1 / (4 * math.pi * physical_constants['vacuum electric permittivity'][0])

    fem_over_fg = (k_e * e**2) / (G_val * m_p * m_e)
    R_res = fem_over_fg / ideal_routed

    print("5. PHYSICAL CROSS-CHECKS & RESIDUE")
    print(f"Observed F_EM / F_G (p-e) ≈ {fem_over_fg:.6e}")
    print(f"Residue factor R_res = {R_res:.6f} (~0.129)")
    print(f"Full relation: F_EM/F_G ≈ [½ × c⁴/8πG × α] × R_res")
    print("-" * 80)

    # 6. Summary & Export
    results = {
        "timestamp": datetime.now().isoformat(),
        "theta_plus_deg": theta_plus_deg,
        "theta_minus_deg": theta_minus_deg,
        "delta_deg": delta_deg,
        "geometric_factor": float(geometric_factor),
        "macro_stiffness": float(macro_stiffness),
        "ideal_routed_scale": float(ideal_routed),
        "log10_ideal": log10_ideal,
        "em_gravity_ratio": float(fem_over_fg),
        "residue_factor_Rres": float(R_res),
        "framework_status": "High-precision tetrahedral bridge with explicit residue",
        "t0c_integration": "θ values aligned to T0C Registry precision (109.47122063449069°)"
    }

    with open("moving_space_results_t0c.json", "w") as f:
        json.dump(results, f, indent=2)

    print("✅ SIMULATION COMPLETE (T0C-Integrated)")
    print("Results exported to moving_space_results_t0c.json\n")

    return results

def plot_ringdown_damping(R_res: float):
    """
    Simulates and plots the gravitational wave ringdown damping profiles.
    """
    # Simulation Parameters
    TIME_END = 0.1  # seconds
    NUM_POINTS = 500
    GR_DAMPING_RATE = 50.0
    INITIAL_AMPLITUDE = 1.0

    time_steps = np.linspace(0, TIME_END, NUM_POINTS)

    # Generate waveforms
    gr_signal = INITIAL_AMPLITUDE * np.exp(-GR_DAMPING_RATE * time_steps)
    modified_damping_rate = GR_DAMPING_RATE * (1.0 + R_res)
    modified_signal = INITIAL_AMPLITUDE * np.exp(-modified_damping_rate * time_steps)

    # Visualization Layout setup
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(11, 6.5))

    ax.plot(time_steps * 1000, gr_signal, label='Standard GR Damping', color='skyblue', linewidth=2)
    ax.plot(time_steps * 1000, modified_signal, label=f'Modified Damping (with $R_{{res}}={R_res:.6f}$)',
            color='salmon', linestyle='--', linewidth=2)

    ax.set_title('Conceptual Ringdown Damping: Standard GR vs. Moving Space Framework', fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Time (ms)', fontsize=11, labelpad=10)
    ax.set_ylabel('Gravitational Wave Strain Amplitude (Normalized)', fontsize=11, labelpad=10)

    ax.legend(fontsize=10, loc='lower left', framealpha=0.2)
    ax.grid(True, linestyle=':', alpha=0.4)

    # Text Annotation
    annotation_text = f"$R_{{res}}$ acts as a local vacuum polarization cutoff,\nleading to increased energy dissipation."
    ax.text(0.60, 0.85, annotation_text, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.6", fc="#2e2b14", ec="#6b6124", alpha=0.8))

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Execute routing computation dynamically
    t0c_data = run_high_precision_routing()

    # Extract calculated residue factor directly into the visualization engine
    plot_ringdown_damping(R_res=t0c_data["residue_factor_Rres"])

# @title
import math
import numpy as np
from scipy.constants import physical_constants

def run_phase_geometry_simulation():
    # Core Constants from T'Z0C White Papers
    GOLDEN_ANGLE = 360.0 / (((1 + math.sqrt(5)) / 2) ** 2) # ~137.5077 degrees
    FINE_STRUCTURE_CONSTANT_INV = 1.0 / physical_constants['fine-structure constant'][0]

    print("====================================================")
    print("      T'Z0C PHASE GEOMETRY VERIFICATION LOOP        ")
    print("====================================================")
    print(f"Target Golden Angle: {GOLDEN_ANGLE:.4f}°")
    print(f"Target Physical Alpha^-1: {FINE_STRUCTURE_CONSTANT_INV:.4f}")
    print("----------------------------------------------------")

    accumulated_phase = 0.0

    # Trace the wraps up to the 137th step threshold
    for wrap in range(1, 138):
        accumulated_phase += GOLDEN_ANGLE
        # Normalize the phase within a standard 360-degree rotation
        normalized_phase = accumulated_phase % 360.0

        # Highlight significant Fibonacci nodes and the final 137 limit
        if wrap in [5, 8, 13, 21, 34, 55, 89, 137]:
            # Calculate distance from the phase center (normalized radius)
            # In Phase Geometry, scale increases as a function of the wrap index
            radius_from_center = math.sqrt(wrap)

            print(f"Wrap #{wrap:3d} (Fibonacci/Limit Node):")
            print(f"  -> Total Accumulated Rotation: {accumulated_phase:10.2f}°")
            print(f"  -> Face-to-Center Phase Vector: {normalized_phase:10.4f}°")
            print(f"  -> Normalized Center Distance:  {radius_from_center:10.4f} units")

            if wrap == 137:
                print("----------------------------------------------------")
                # Incorporate the Thomas Precession / Wigner Rotation factor (1/2)
                # This factor arises from the non-commutativity of Lorentz boosts (Wigner rotation)
                # and accounts for the relativistic correction in spin-orbit coupling,
                # where a 'naive' calculation is found to be a factor of 2 too large.
                # Geometrically, it aligns with the 'geometric_factor' (0.5) derived from T'Z₀C tetrahedral geometry.
                THOMAS_FACTOR = 0.5

                # Here, GOLDEN_ANGLE is treated as a 'naive' theoretical value, analogous to
                # how a factor of 2 was historically found to be too large in certain
                # relativistic coupling calculations *before* accounting for Thomas precession.
                # It is NOT a direct theoretical derivation of Alpha^-1 itself.
                NAIVE_THEORETICAL_ALPHA_INV = GOLDEN_ANGLE

                # Apply the Thomas Factor to correct this naive theoretical value
                THOMAS_CORRECTED_THEORETICAL_ALPHA_INV = NAIVE_THEORETICAL_ALPHA_INV * THOMAS_FACTOR

                # Compare the Thomas-corrected theoretical value to the target physical value
                corrected_angular_discrepancy = abs(THOMAS_CORRECTED_THEORETICAL_ALPHA_INV - FINE_STRUCTURE_CONSTANT_INV)

                # Original angular discrepancy (for comparison, without Thomas Factor)
                original_angular_discrepancy = abs(NAIVE_THEORETICAL_ALPHA_INV - FINE_STRUCTURE_CONSTANT_INV)

                # Calculate coherence fidelity based on the corrected value
                # A smaller 'corrected_angular_discrepancy' implies higher fidelity.
                coherence_fidelity_corrected = (1 - (corrected_angular_discrepancy / FINE_STRUCTURE_CONSTANT_INV)) * 100
                coherence_fidelity_original = (1 - (original_angular_discrepancy / FINE_STRUCTURE_CONSTANT_INV)) * 100

                print(f"CRITICAL SATURATION ANALYSIS AT 137th WRAP (Thomas Precession Integrated):")
                print(f"  -> Naive Theoretical Alpha^-1 (from Golden Angle): {NAIVE_THEORETICAL_ALPHA_INV:.4f}°")
                print(f"  -> Physical Alpha^-1 (Target):                    {FINE_STRUCTURE_CONSTANT_INV:.4f}")
                print(f"  -> Thomas Precession Factor:                      {THOMAS_FACTOR:.1f} (from T'Z₀C geometry)")
                print(f"  -> Thomas-Corrected Theoretical Alpha^-1:         {THOMAS_CORRECTED_THEORETICAL_ALPHA_INV:.4f}°")
                print(f"  -> Original Angular Discrepancy:                  {original_angular_discrepancy:.4f}°")
                print(f"  -> Corrected Angular Discrepancy (with Thomas Factor): {corrected_angular_discrepancy:.4f}°")
                print(f"  -> Original Magnetic Coherence Fidelity:          {coherence_fidelity_original:.2f}%")
                print(f"  -> Corrected Magnetic Coherence Fidelity:         {coherence_fidelity_corrected:.2f}% (decreased fidelity)")
                print("  (This demonstrates that while the Thomas Factor is a crucial relativistic correction")
                print("   arising from the non-commutative nature of Lorentz boosts (Wigner rotation) and is")
                print("   historically significant for correcting terms like spin-orbit coupling, its direct")
                print("   scalar application to the Golden Angle does not improve the approximation of the")
                print("   inverse fine-structure constant. The white paper explicitly states it fixes a factor")
                print("   of 2 in coupling coefficients, not in the derivation of alpha itself.)")

if __name__ == "__main__":
    run_phase_geometry_simulation()
